In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("/kaggle/input/datasets/hussnainmamoon1/llm-hallucination-benchmark-v2-25k-rows-6-models/llm_hallucination_benchmark_v2_research_grade (1).csv")

In [31]:
# Check columns
print(df.shape)
print(df["hallucination_type"].value_counts())

(25200, 24)
hallucination_type
fabrication    15216
none            7244
confusion       2740
Name: count, dtype: int64


In [32]:
df = df[["prompt_text", "hallucination_type"]]

df = df[df["hallucination_type"].isin(["none","fabrication","confusion"])]

print("Original class counts: ")
print(df["hallucination_type"].value_counts())

Original class counts: 
hallucination_type
fabrication    15216
none            7244
confusion       2740
Name: count, dtype: int64


In [33]:
# # Create balanced subset: 2000 rows per class
# subset_df = (
#     df.groupby("hallucination_type", group_keys=False)
#       .apply(lambda x: x.sample(n=2000, random_state=42))
#       .reset_index(drop=True)
# )

# # Shuffle the subset
# subset_df = subset_df.sample(frac=1, random_state=42).reset_index(drop=True)

# # Encode labels
# label_map = {
#     "none": 0,
#     "fabrication": 1,
#     "confusion": 2
# }

# subset_df["label"] = subset_df["hallucination_type"].map(label_map)

# # Save subset
# subset_df.to_csv("hallucination_subset_6000.csv", index=False)

# print("Subset saved successfully!")
# print(subset_df.shape)
# print(subset_df["hallucination_type"].value_counts())

In [34]:
label_map = {
    "none": 0,
    "fabrication": 1,
    "confusion": 2
}

df["label"] = df["hallucination_type"].map(label_map)

print(df.shape)
print(df["hallucination_type"].value_counts())

(25200, 3)
hallucination_type
fabrication    15216
none            7244
confusion       2740
Name: count, dtype: int64


In [35]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

train_df.to_csv("train_full.csv", index=False)
val_df.to_csv("validation_full.csv", index=False)
test_df.to_csv("test_full.csv", index=False)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (20160, 3)
Validation: (2520, 3)
Test: (2520, 3)


In [36]:
!pip install transformers datasets evaluate -q

In [37]:
from datasets import Dataset
from transformers import AutoTokenizer

# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Convert pandas dataframes to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["prompt_text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Keep only columns needed for training
train_dataset = train_dataset.remove_columns(["prompt_text", "hallucination_type"])
val_dataset = val_dataset.remove_columns(["prompt_text", "hallucination_type"])
test_dataset = test_dataset.remove_columns(["prompt_text", "hallucination_type"])

# Rename label column
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

# Set format for PyTorch
train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

print(train_dataset)
print(val_dataset)
print(test_dataset)

Map:   0%|          | 0/20160 [00:00<?, ? examples/s]

Map:   0%|          | 0/2520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2520 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 20160
})
Dataset({
    features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2520
})
Dataset({
    features: ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2520
})


In [38]:
!pip install dagshub mlflow -q

In [39]:
import os
import mlflow
import dagshub

# Add a secret named DAGSHUB_TOKEN in Kaggle
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN1")

os.environ["MLFLOW_TRACKING_USERNAME"] = "ArpitaMallik"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

In [40]:
dagshub.init(
    repo_owner="ArpitaMallik",
    repo_name="mlops-llm-hallucination-type-detection-project",
    mlflow=True
)

mlflow.set_tracking_uri(
    "https://dagshub.com/ArpitaMallik/mlops-llm-hallucination-type-detection-project.mlflow"
)

mlflow.set_experiment("DistilBERT Baseline")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=be1a4285-718b-4f7b-aac7-52b533a80524&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=ebbf7a7c1f7eb904911e397f8d846ebfdb084f2336c52c5fd51d86326474e75b




Accessing as ArpitaMallik

Initialized MLflow to track repo "ArpitaMallik/mlops-llm-hallucination-type-detection-project"

Repository ArpitaMallik/mlops-llm-hallucination-type-detection-project initialized!

2026/06/03 19:58:37 INFO mlflow.tracking.fluent: Experiment with name 'DistilBERT Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/c58bb7d2a3134aaa9e57a6bfe60631c8', creation_time=1780516717406, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1780516717406, lifecycle_stage='active', name='DistilBERT Baseline', tags={}, trace_location=None, workspace='default'>

In [41]:
with mlflow.start_run():
    mlflow.log_param("model_name", "distilbert-base-uncased")
    mlflow.log_param("epochs", 2)
    mlflow.log_param("max_length", 256)
    mlflow.log_param("batch_size", 16)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("macro_f1", f1_macro)
    mlflow.log_metric("weighted_f1", f1_weighted)

🏃 View run overjoyed-croc-695 at: https://dagshub.com/ArpitaMallik/mlops-llm-hallucination-type-detection-project.mlflow/#/experiments/0/runs/599d38cec43846d59bf1f844ad466dce
🧪 View experiment at: https://dagshub.com/ArpitaMallik/mlops-llm-hallucination-type-detection-project.mlflow/#/experiments/0


In [42]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [43]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    acc = accuracy.compute(
        predictions=predictions,
        references=labels
    )

    f1_score = f1.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

In [44]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert_hallucination_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [45]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.588699,1.649902,0.625000,0.591177
2,1.633396,1.626500,0.624206,0.591297
3,1.614268,1.620778,0.626587,0.592545


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1890, training_loss=1.642012306495949, metrics={'train_runtime': 752.2705, 'train_samples_per_second': 80.397, 'train_steps_per_second': 2.512, 'total_flos': 4005885573365760.0, 'train_loss': 1.642012306495949, 'epoch': 3.0})

In [46]:
trainer.evaluate(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.6189101934432983,
 'eval_accuracy': 0.6305555555555555,
 'eval_f1': 0.5963409410402027,
 'eval_runtime': 10.0537,
 'eval_samples_per_second': 250.655,
 'eval_steps_per_second': 7.858,
 'epoch': 3.0}

In [47]:
import mlflow

with mlflow.start_run(run_name="distilbert_baseline"):

    mlflow.log_param("model_name", "distilbert-base-uncased")
    mlflow.log_param("epochs", 3)
    mlflow.log_param("batch_size", 16)
    mlflow.log_param("learning_rate", 2e-5)
    mlflow.log_param("max_length", 256)

    trainer.train()

    test_results = trainer.evaluate(test_dataset)

    mlflow.log_metrics(test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.572039,1.639875,0.626587,0.594756
2,1.624666,1.624823,0.625000,0.592261
3,1.609282,1.620301,0.621429,0.586985


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


🏃 View run distilbert_baseline at: https://dagshub.com/ArpitaMallik/mlops-llm-hallucination-type-detection-project.mlflow/#/experiments/0/runs/797aa92257db4e8d9a16b1165249c45e
🧪 View experiment at: https://dagshub.com/ArpitaMallik/mlops-llm-hallucination-type-detection-project.mlflow/#/experiments/0


In [48]:
# Save model and tokenizer
trainer.save_model("saved_distilbert_hallucination_model")
tokenizer.save_pretrained("saved_distilbert_hallucination_model")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [49]:
import shutil

shutil.make_archive(
    "saved_distilbert_hallucination_model",
    "zip",
    "saved_distilbert_hallucination_model"
)

print("Model zipped successfully!")
print("File name: saved_distilbert_hallucination_model.zip")

Model zipped successfully!
File name: saved_distilbert_hallucination_model.zip


In [50]:
import torch

id2label = {
    0: "none",
    1: "fabrication",
    2: "confusion"
}

def predict_hallucination_type(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    return id2label[prediction]

In [51]:
sample_text = "What is the role of insulin in the human body?"

prediction = predict_hallucination_type(sample_text)

print("Predicted hallucination type:", prediction)

Predicted hallucination type: fabrication


In [52]:
test_sample = test_df.iloc[0]["prompt_text"]

print("Prompt:", test_sample)
print("Actual:", test_df.iloc[0]["hallucination_type"])
print("Predicted:", predict_hallucination_type(test_sample))

Prompt: What is a placebo?
Actual: none
Predicted: none


In [53]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Get predictions
predictions = trainer.predict(test_dataset)

# Convert logits to predicted labels
y_pred = np.argmax(predictions.predictions, axis=1)

# True labels
y_true = predictions.label_ids

# Class names
class_names = ["none", "fabrication", "confusion"]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [54]:
# Basic metrics
accuracy = accuracy_score(y_true, y_pred)

precision_macro = precision_score(y_true, y_pred, average="macro")
recall_macro = recall_score(y_true, y_pred, average="macro")
f1_macro = f1_score(y_true, y_pred, average="macro")

precision_weighted = precision_score(y_true, y_pred, average="weighted")
recall_weighted = recall_score(y_true, y_pred, average="weighted")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

print("Accuracy:", accuracy)

print("Macro Precision:", precision_macro)
print("Macro Recall:", recall_macro)
print("Macro F1:", f1_macro)

print("Weighted Precision:", precision_weighted)
print("Weighted Recall:", recall_weighted)
print("Weighted F1:", f1_weighted)

Accuracy: 0.6285714285714286
Macro Precision: 0.40286359072007966
Macro Recall: 0.4635982074708865
Macro F1: 0.429122364943765
Weighted Precision: 0.5720223113491686
Weighted Recall: 0.6285714285714286
Weighted F1: 0.5967319827465947


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [55]:
# Detailed class-wise report
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

              precision    recall  f1-score   support

        none       0.50      0.67      0.57       725
 fabrication       0.71      0.72      0.72      1521
   confusion       0.00      0.00      0.00       274

    accuracy                           0.63      2520
   macro avg       0.40      0.46      0.43      2520
weighted avg       0.57      0.63      0.60      2520



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [56]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print(cm)

[[ 484  241    0]
 [ 421 1100    0]
 [  66  208    0]]
